# 🐛 CrewAI Code Debugger

[![Python](https://img.shields.io/badge/Python-3.10%2B-blue?logo=python&logoColor=white)](https://www.python.org/)
[![CrewAI](https://img.shields.io/badge/CrewAI-1.12%2B-orange?logo=robot&logoColor=white)](https://github.com/crewAIInc/crewAI)
[![OpenAI](https://img.shields.io/badge/OpenAI-GPT--4o-412991?logo=openai&logoColor=white)](https://platform.openai.com/)
[![License](https://img.shields.io/badge/License-MIT-green)](LICENSE)

**Objective:** Build a multi-agent code debugging system using CrewAI's sequential process that analyzes Python code for syntax and logical errors, then automatically corrects them — powered by OpenAI GPT-4o.

> Workflow diagram: [`Flow/workflow.mmd`](Flow/workflow.mmd)

---

## 🤖 Agent Descriptions

| Agent | Role | CrewAI Class |
|-------|------|--------------|
| **Code Analyzer** | Identifies all syntax and logical errors in the provided Python code using `CodeInterpreterTool` | `Agent` |
| **Code Corrector** | Fixes every identified error and returns clean, PEP-8 compliant Python code | `Agent` |
| **Manager** | Oversees the analysis and correction pipeline, coordinating delegation and quality | `Agent` |

---

## 🔄 Pipeline Flow

1. Buggy Python code is embedded directly in the `Analysis Task` description
2. `Process.sequential` routes the task to **Code Analyzer** first
3. The analyzer runs the code via `CodeInterpreterTool` and outputs a numbered error list
4. The **Code Corrector** receives the error list as context and fixes every issue
5. The **Manager** agent oversees both steps with `allow_delegation=True`
6. `planning=True` enables the crew to coordinate steps before execution begins

---

## Table of Contents
1. [Imports & Environment Setup](#1-imports--environment-setup)
2. [LLM Configuration — GPT-4o](#2-llm-configuration)
3. [Define Buggy Input Code](#3-define-buggy-input-code)
4. [Define Agents](#4-define-agents)
5. [Define Tasks](#5-define-tasks)
6. [Build the Crew](#6-build-the-crew)
7. [Run the Pipeline](#7-run-the-pipeline)
8. [Agent Flow Summary](#8-agent-flow-summary)

---
## 1. Imports & Environment Setup

All dependencies are listed in [`requirements.txt`](requirements.txt). Install with:
```bash
pip install -r requirements.txt
```

Set your OpenAI API key in `.env`:
```
OPENAI_API_KEY=your_openai_api_key_here
OPENAI_MODEL_NAME=gpt-4o
```

> **Note:** `CodeInterpreterTool` executes code inside a Docker container. Make sure Docker Desktop is running before executing the pipeline cell.

In [ ]:
import os
from datetime import datetime

from crewai import Agent, Task, Crew, Process
from crewai_tools import CodeInterpreterTool
from dotenv import load_dotenv
from IPython.display import display, Markdown

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
MODEL = os.getenv("OPENAI_MODEL_NAME", "gpt-4o")

print("✅ Libraries imported successfully!")
print(f"📅 Session started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
if OPENAI_API_KEY and OPENAI_API_KEY != "your_openai_api_key_here":
    masked = '*' * (len(OPENAI_API_KEY) - 4) + OPENAI_API_KEY[-4:]
    print(f"🔑 OPENAI_API_KEY loaded: {masked}")
else:
    print("⚠️  OPENAI_API_KEY not set — add it to your .env file before running the pipeline.")
print(f"🤖 Model: {MODEL}")

---
## 2. LLM Configuration

CrewAI automatically reads `OPENAI_API_KEY` and `OPENAI_MODEL_NAME` from the environment — no explicit client setup is needed.

The `CodeInterpreterTool` runs submitted Python code inside an **isolated Docker container** and returns the stdout output, making it safe for executing untrusted or buggy code.

In [ ]:
# ─── Instantiate the CodeInterpreterTool ─────────────────────────────────────
# Executes Python3 code in an isolated Docker container and captures stdout.
# Docker Desktop must be running for this tool to work.
code_interpreter = CodeInterpreterTool()

print("✅ LLM & tool configuration ready!")
print(f"   Model    : {MODEL}")
print(f"   Provider : OpenAI (via OPENAI_API_KEY env var)")
print(f"   Tool     : CodeInterpreterTool (Docker-based Python executor)")

---
## 3. Define Buggy Input Code

This is the Python snippet the crew will analyze and fix.
The `fibonacci_iterative` function has **indentation errors** on every line inside the function body — none of the `if/elif/for/return` blocks are indented correctly.

> To debug your own code, replace the `BUGGY_CODE` string below with your snippet.

In [ ]:
# ================================================================
# BUGGY PYTHON CODE — input for the agent pipeline
# ================================================================
BUGGY_CODE = """\
def fibonacci_iterative(n):
if n < 0:
return []
elif n == 1:
return [0]
elif n == 2:
return [0, 1]
fib_sequence = [0, 1]
for i in range(2, n):
next_fib = fib_sequence[-1] + fib_sequence[-2]
fib_sequence.append(next_fib)
return fib_sequence
"""

print("✅ Buggy code loaded!")
print()
print("─" * 52)
print(BUGGY_CODE)
print("─" * 52)
print("⚠️  Known issues: indentation errors throughout the function body")

---
## 4. Define Agents

Each agent is a CrewAI `Agent` with a defined `role`, `goal`, and `backstory`.

- **Code Analyzer** is equipped with `CodeInterpreterTool` to execute and evaluate the code at runtime.
- **Code Corrector** works purely from the error report — no tools needed.
- **Manager** has `allow_delegation=True` to coordinate the other two agents and oversee quality.

In [ ]:
# ================================================================
# AGENT 1: CODE ANALYZER
# Role: Identify all syntax and logical errors using CodeInterpreterTool
# ================================================================
code_analyzer = Agent(
    role="Code Analyzer",
    goal="Identify all syntax and logical errors in the provided Python code.",
    backstory=(
        "You are an expert Python developer with deep knowledge of syntax rules, "
        "indentation requirements, and common logical pitfalls. You use the "
        "CodeInterpreterTool to execute and evaluate code, then produce a clear, "
        "numbered list of every error found."
    ),
    tools=[code_interpreter],
    verbose=True,
    allow_delegation=False,
)

print("✅ Code Analyzer agent defined!")


# ================================================================
# AGENT 2: CODE CORRECTOR
# Role: Fix every error identified by the Code Analyzer
# ================================================================
code_corrector = Agent(
    role="Code Corrector",
    goal="Fix every error identified by the Code Analyzer and return clean, working Python code.",
    backstory=(
        "You are a meticulous Python engineer who takes error reports and produces "
        "corrected, well-formatted code. You apply PEP-8 standards and ensure the "
        "logic is sound before delivering the final output."
    ),
    verbose=True,
    allow_delegation=False,
)

print("✅ Code Corrector agent defined!")


# ================================================================
# AGENT 3: MANAGER
# Role: Oversee the pipeline and coordinate delegation
# ================================================================
manager = Agent(
    role="Manager",
    goal="Oversee the code analysis and correction workflow, ensuring quality output.",
    backstory=(
        "You are a senior engineering lead responsible for coordinating the debugging "
        "pipeline. You ensure the analyzer and corrector complete their tasks correctly "
        "and that the final output meets production standards."
    ),
    verbose=True,
    allow_delegation=True,
)

print("✅ Manager agent defined!")
print()
print("   Code Analyzer  → identifies errors via CodeInterpreterTool")
print("   Code Corrector → fixes errors, returns clean PEP-8 code")
print("   Manager        → oversees pipeline, delegates as needed")

---
## 5. Define Tasks

Tasks are the units of work assigned to each agent.

- **Analysis Task** passes the buggy code to the Code Analyzer via `CodeInterpreterTool`.
- **Correction Task** uses `context=[analysis_task]` so the corrector automatically receives the full error report as input — no manual passing needed.

In [ ]:
# ================================================================
# TASK 1: ANALYSIS TASK
# Agent: Code Analyzer | Tool: CodeInterpreterTool
# ================================================================
analysis_task = Task(
    description=(
        f"Analyze the following Python code using the CodeInterpreterTool. "
        f"Identify ALL syntax errors (e.g., indentation issues, missing colons) "
        f"and logical errors. Return a numbered list of every error found.\n\n"
        f"```python\n{BUGGY_CODE}\n```"
    ),
    expected_output=(
        "A numbered list of all errors found in the code, specifying the error type "
        "(syntax/logical), the line or block affected, and a brief description."
    ),
    agent=code_analyzer,
)

print("✅ Analysis Task defined!")


# ================================================================
# TASK 2: CORRECTION TASK
# Agent: Code Corrector | Context: analysis_task output
# ================================================================
correction_task = Task(
    description=(
        "Using the error list from the Code Analyzer, fix every identified issue in "
        "the original Python code. Return ONLY the fully corrected Python code block, "
        "properly indented and formatted to PEP-8 standards."
    ),
    expected_output=(
        "The complete, corrected Python code with all syntax and logical errors resolved, "
        "ready to run without modification."
    ),
    agent=code_corrector,
    context=[analysis_task],
)

print("✅ Correction Task defined!")
print()
print("   Analysis Task   → Code Analyzer  (CodeInterpreterTool)")
print("   Correction Task → Code Corrector (context: analysis_task output)")

---
## 6. Build the Crew

`Process.sequential` runs tasks in the order they are listed — Analysis first, then Correction.

Key crew settings:
- `manager_agent=manager` — assigns the Manager to oversee the pipeline
- `planning=True` — enables a pre-execution planning phase where the Manager coordinates steps before any agent starts working
- `verbose=True` — streams each agent's reasoning and tool calls to the output in real time

In [ ]:
# ─── Assemble the Crew ────────────────────────────────────────────────────────
crew = Crew(
    agents=[code_analyzer, code_corrector],
    tasks=[analysis_task, correction_task],
    process=Process.sequential,
    manager_agent=manager,
    planning=True,
    verbose=True,
)

print("✅ Crew assembled!")
print(f"   Agents   : {[a.role for a in crew.agents]}")
print(f"   Tasks    : {len(crew.tasks)}")
print(f"   Process  : {crew.process}")
print(f"   Planning : {crew.planning}")
print(f"   Manager  : {crew.manager_agent.role}")

---
## 7. Run the Pipeline

`crew.kickoff()` starts the sequential pipeline:

1. **Manager** runs the planning phase — coordinates the full execution plan
2. **Code Analyzer** executes the buggy code via `CodeInterpreterTool` and returns a numbered error list
3. **Code Corrector** receives the error list as context and returns the fully fixed, PEP-8 code

> Ensure Docker Desktop is running and `OPENAI_API_KEY` is set in `.env` before executing this cell.

In [ ]:
print("🚀 Starting CrewAI Code Debugger pipeline...")
print("=" * 60)

result = crew.kickoff()

print("=" * 60)
print("✅ Debugging pipeline complete!")
print()
print("─── Final Output ───────────────────────────────────────────")
print(result)
print("────────────────────────────────────────────────────────────")

---
## 8. Agent Flow Summary

In [ ]:
summary = """
## 🗨️ CrewAI Code Debugger — Sequential Crew Flow

| Step | Agent | Role | CrewAI Class |
|------|-------|------|--------------|
| 1 | Manager (planning phase) | Plans and coordinates the full pipeline before execution | `Agent` (allow_delegation=True) |
| 2 | Code Analyzer | Runs buggy code via CodeInterpreterTool, outputs numbered error list | `Agent` + `CodeInterpreterTool` |
| 3 | Code Corrector | Receives error list as context, returns corrected PEP-8 code | `Agent` (context=[analysis_task]) |

### CrewAI Components Used
- `crewai.Agent` — Code Analyzer, Code Corrector, and Manager agents
- `crewai.Task` — Analysis Task and Correction Task with context chaining
- `crewai.Crew` — orchestrates agents and tasks with `Process.sequential`
- `crewai.Process.sequential` — ensures Analysis runs before Correction
- `crewai_tools.CodeInterpreterTool` — executes Python code in an isolated Docker container
- `planning=True` — enables pre-execution planning by the Manager agent

### LLM
- Model    : `gpt-4o` (configurable via `OPENAI_MODEL_NAME` env var)
- Provider : OpenAI
- Config   : environment variables (`OPENAI_API_KEY`, `OPENAI_MODEL_NAME`)

### Repository Structure
```
crewai-code-debugger/
├── crewai_debugger.ipynb   # This notebook
├── Flow/
│   └── workflow.mmd        # Mermaid workflow diagram
├── requirements.txt
├── .env.example
├── .gitignore
└── LICENSE
```
"""
display(Markdown(summary))